In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_real_estate_valuation_thesis(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "RealEstateValuationId String, SourceSystem string, PortfolioKey string, PortfolioName string, IngestionTimestamp timestamp,"
        #     "BatchId string, RealEstateValuationKey string, RealEstateKey String, RealEstateDescription string, RealEstateTypeCode string,"
        #     "RealEstateType string, RealEstateValuationStatusCode string, RealEstateValuationStatus string, RealEstateValuatorKey string,"
        #     "RealEstateValuationType1 string, RealEstateValuationType2 string, ValuationPurpose string, ValuationMethod string,"
        #     "ValuationMethodWeights string, MarketabilityIndex string, AutopsyScope string, AutopsyDate timestamp, ValuationDate timestamp,"
        #     "MarketValue Decimal(19,4), LeaseValue Decimal(19,4), InsuranceValue Decimal(19,4), ReasonableValue Decimal(19,4),"
        #     "FiscalValue Decimal(19,4), ReplacementValue Decimal(19,4), EquipmentValue Decimal(19,4), ValuationCost Decimal(19,4),"
        #     "OtherCost Decimal(19,4), Address1 string, Address2 string, ZipCode string, City string, Area string, Geolocation string,"
        #     "DetectionProblemFlag boolean, AreaTopographic double, AreaTitle double, AreaComments string,"
        #     "BuildingFactor double, FeasibleFactor double, CoverageFactor double, AreaEven int, AreaMin int,"
        #     "FacadeMin int, MaxHeight double, BuildingIntroDescription string, BuildingPermit string,"
        #     "BuildingReadiness double, CompletionYear int, Comments string, EnteredBy int, DateEntered timestamp, TimeToSell int,"
        #     "IsActiveFlag boolean, ActionCode int, HtmlRealTitle string, HtmlDocuments string, HtmlLand string, HtmlLimitations string,"
        #     "HtmlAlloweduse string, HtmlDeclaration string, HtmlBuildingurban string, HtmlBuildinglegal string, HtmlAssignor string,"
        #     "HtmlAttention string, HtmlOwnershipHeader string, HtmlOwnershipFooter string, BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.RealEstateValuation"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "thesis",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== Portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
            "businessdate",
            "portfolio_id_thesis",
        ]

        portfolio = load_landed_portfolio_table(
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== prvaluation ==================

        prvaluation_extra_filters = None

        if prvaluation_extra_filters is not None:
            prvaluation_extra_filters.append(date_filter)
            prvaluation_final_filters = " AND ".join(prvaluation_extra_filters)
        else:
            prvaluation_final_filters = date_filter

        prvaluation_selected_fields = [
            "id",
            "portfolio_assignor_id",
            "property_id",
            "property_descr",
            "property_type",
            "status_code",
            "valuator_id",
            "valuation_type",
            "type_code",
            "purpose_code",
            "valuation_method",
            "html_method_weights",
            "marketability_code",
            "autopsy",
            "autopsy_date",
            "valuation_date",
            "market_value",
            "lease_value",
            "insurance_value",
            "reasonable_value",
            "fiscal_value",
            "replacement_value",
            "equipment_value",
            "valuation_cost",
            "other_cost",
            "address",
            "address_2",
            "zip",
            "city",
            "area",
            "geolocation",
            "detection_problem",
            "area_topo",
            "area_title",
            "comments_area",
            "factor_building",
            "factor_feasible",
            "factor_coverage",
            "area_even",
            "area_min",
            "facade_min",
            "max_height",
            "html_building_intro",
            "html_building_permit",
            "building_ready",
            "completion_year",
            "comments",
            "entered_by",
            "date_entered",
            "time_to_sell",
            "active",
            "action_id",
            "html_realtitle",
            "html_documents",
            "html_land",
            "html_limitations",
            "html_allowed_use",
            "html_declaration",
            "html_building_urban",
            "html_building_legal",
            "html_assignor",
            "html_attention",
            "html_ownership_header",
            "html_ownership_footer",
            "businessdate",
        ]

        prvaluation = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "thesis_prvaluation",
            prvaluation_final_filters,
            prvaluation_selected_fields,
        )
        # Join Condition
        Portfolio_Prvaluation_join_condition = (
            portfolio["portfolio_id_thesis"] == prvaluation["portfolio_assignor_id"]
        )

        # Join
        result_df = prvaluation.join(
            portfolio,
            Portfolio_Prvaluation_join_condition,
            "left",
        )

        prvaluation_primary_key_fields = [
            prvaluation.id,
            lit("thesis"),
        ]

        result_df = result_df.withColumn(
            "RealEstateValuationId",
            sha2(concat_ws(";", *prvaluation_primary_key_fields), 256),
        )

        # ================== Selection of Final Columns ==================

        result_df = result_df.select(
            result_df.RealEstateValuationId.alias("RealEstateValuationId"),
            lit("thesis").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            prvaluation.portfolio_assignor_id.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.alias("PortfolioName"),
            # Use BusinessDate of prvaluation
            # lit(current_timestamp()).alias("IngestionDate"),
            lit("test BatchId").alias("BatchId"),  # Not available yet
            prvaluation.id.cast("string").alias("RealEstateValuationKey"),
            prvaluation.property_id.cast("string").alias("RealEstateKey"),
            prvaluation.property_descr.alias("RealEstateDescription"),
            prvaluation.property_type.alias(
                "RealEstateTypeCode"
            ),  # Description values (mostly, we have some numbers as well)
            prvaluation.property_type.alias(
                "RealEstateType"
            ),  # Same as above field because it is already description
            prvaluation.status_code.alias(
                "RealEstateValuationStatusCode"
            ),  # Description values
            prvaluation.status_code.alias(
                "RealEstateValuationStatus"
            ),  # Same as above field because it is already description
            prvaluation.valuator_id.cast("string").alias("RealEstateValuatorKey"),
            prvaluation.valuation_type.alias("RealEstateValuationType1"),
            prvaluation.type_code.alias("RealEstateValuationType2"),
            prvaluation.purpose_code.alias("ValuationPurpose"),
            prvaluation.valuation_method.alias("ValuationMethod"),
            prvaluation.html_method_weights.alias("ValuationMethodWeights"),
            prvaluation.marketability_code.alias("MarketabilityIndex"),
            prvaluation.autopsy.alias("AutopsyScope"),
            prvaluation.autopsy_date.alias("AutopsyDate"),
            prvaluation.valuation_date.alias("ValuationDate"),
            prvaluation.market_value.alias("MarketValue"),
            prvaluation.lease_value.alias("LeaseValue"),
            prvaluation.insurance_value.alias("InsuranceValue"),
            prvaluation.reasonable_value.alias("ReasonableValue"),
            prvaluation.fiscal_value.alias("FiscalValue"),
            prvaluation.replacement_value.alias("ReplacementValue"),
            prvaluation.equipment_value.alias("EquipmentValue"),
            prvaluation.valuation_cost.alias("ValuationCost"),
            prvaluation.other_cost.alias("OtherCost"),
            prvaluation.address.alias("Address1"),
            prvaluation.address_2.alias("Address2"),
            prvaluation.zip.alias("ZipCode"),
            prvaluation.city.alias("City"),
            prvaluation.area.alias("Area"),
            prvaluation.geolocation.alias("Geolocation"),
            prvaluation.detection_problem.alias("DetectionProblemFlag"),
            prvaluation.area_topo.alias("AreaTopographic"),
            prvaluation.area_title.alias("AreaTitle"),
            prvaluation.comments_area.alias("AreaComments"),
            prvaluation.factor_building.alias("BuildingFactor"),
            prvaluation.factor_feasible.alias("FeasibleFactor"),
            prvaluation.factor_coverage.alias("CoverageFactor"),
            prvaluation.area_even.alias("AreaEven"),
            prvaluation.area_min.alias("AreaMin"),
            prvaluation.facade_min.alias("FacadeMin"),
            prvaluation.max_height.cast("double").alias("MaxHeight"),
            prvaluation.html_building_intro.alias("BuildingIntroDescription"),
            prvaluation.html_building_permit.alias("BuildingPermit"),
            prvaluation.building_ready.alias("BuildingReadiness"),
            prvaluation.completion_year.alias("CompletionYear"),
            prvaluation.comments.alias("Comments"),
            prvaluation.entered_by.alias("EnteredBy"),
            prvaluation.date_entered.alias("DateEntered"),
            prvaluation.time_to_sell.alias("TimeToSell"),
            prvaluation.active.alias("IsActiveFlag"),
            prvaluation.action_id.alias("ActionCode"),
            prvaluation.html_realtitle.alias("HtmlRealTitle"),
            prvaluation.html_documents.alias("HtmlDocuments"),
            prvaluation.html_land.alias("HtmlLand"),
            prvaluation.html_limitations.alias("HtmlLimitations"),
            prvaluation.html_allowed_use.alias("HtmlAlloweduse"),
            prvaluation.html_declaration.alias("HtmlDeclaration"),
            prvaluation.html_building_urban.alias("HtmlBuildingurban"),
            prvaluation.html_building_legal.alias("HtmlBuildinglegal"),
            prvaluation.html_assignor.alias("HtmlAssignor"),
            prvaluation.html_attention.alias("HtmlAttention"),
            prvaluation.html_ownership_header.alias("HtmlOwnershipHeader"),
            prvaluation.html_ownership_footer.alias("HtmlOwnershipFooter"),
            prvaluation.businessdate.alias("BusinessDate"),
        )

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_real_estate_valuation_thesis(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)